In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json 
import itertools

fsize = 20
plt.rcParams.update({"font.size": fsize})
%config InlineBackend.figure_format = 'retina'

In [2]:
def load_evidence(fn):
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    del df["derived"]
    return df

In [3]:
# fn_human = "../../data/adipose_Massier2023/evidence_human/evidence.json"
fn_human = "../../data/adipose_Emont2022/evidence_human/evidence.json"

deg = load_evidence(fn_human)

In [4]:
deg.head()

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type
0,text,We identified six distinct subpopulations of h...,text,homo_sapiens,human ASPCs,adipose,None,PDGFRA,gene
1,text,"For example, mASPC2 and hASPC2 are characteriz...",text,homo_sapiens,hASPC2,adipose,None,ALDH1A3,gene
2,text,"Similarly, mASPC4 and hASPC4 express Epha3 and...",text,homo_sapiens,hASPC4,adipose,None,EPHA3,gene
3,text,"Thus, the adipokines adiponectin and adipsin (...",text,homo_sapiens,hAd3,adipose,None,CFD,gene
4,text,genes encoding insulin signalling components s...,text,homo_sapiens,hAd5,adipose,None,INSR,gene


In [5]:
from extract.extract import align

In [6]:
deg.head()

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type
0,text,We identified six distinct subpopulations of h...,text,homo_sapiens,human ASPCs,adipose,None,PDGFRA,gene
1,text,"For example, mASPC2 and hASPC2 are characteriz...",text,homo_sapiens,hASPC2,adipose,None,ALDH1A3,gene
2,text,"Similarly, mASPC4 and hASPC4 express Epha3 and...",text,homo_sapiens,hASPC4,adipose,None,EPHA3,gene
3,text,"Thus, the adipokines adiponectin and adipsin (...",text,homo_sapiens,hAd3,adipose,None,CFD,gene
4,text,genes encoding insulin signalling components s...,text,homo_sapiens,hAd5,adipose,None,INSR,gene


In [7]:
tdeg = deg.query("source_type == 'text'").copy()

In [8]:
tdeg["source_cell_type_label"] = tdeg.apply(lambda x: align(x["source_rationale"], x["extracted_cell_type_label"] )[0], axis=1)
tdeg["source_feature_name"] = tdeg.apply(lambda x: align(x["source_rationale"], x["extracted_feature_name"] )[0], axis=1)

In [9]:
tdeg["found_feature_name"] = tdeg["source_feature_name"] == tdeg["extracted_feature_name"]
tdeg["found_cell_type_label"] = tdeg["source_cell_type_label"] == tdeg["extracted_cell_type_label"]

In [10]:
print("Features   found: , ", tdeg["found_feature_name"].sum() )
print("Cell types found: , ", tdeg["found_cell_type_label"].sum() )

Features   found: ,  17
Cell types found: ,  17


In [11]:
tdeg.query("~found_cell_type_label")

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,source_cell_type_label,source_feature_name,found_feature_name,found_cell_type_label
